# modeling_v14 — V2 시간축 피처 선택 (GV2 스크리닝)

**로드맵** V2 · **판정 축** rolling-origin time-CV(主) · **벤치마크** core10+재학습 = **98.361 · R² 0.8587**

> ⚠️ **실행 위치**: `modeling_v14/` 폴더 (옆 `v14_features.py` + `data/v14_P*.csv` 사용)
> ⚠️ **미러 미검증**: v14 방침대로 클라우드 실행검증 안 함. 정적 검토 + assert 로 보완.

## 사전등록 (결과 보기 전 고정 · R4)

**후보 6종 — 이 목록 외 ad-hoc 추가 금지(R7)**

| # | 후보 | 피처 |
|---|---|---|
| 1 | core10 (벤치마크 재현) | 10 |
| 2 | core10 + **P1**(PM위상 z) | 10+75 |
| 3 | core10 + **P2**(레짐 baseline 차감) | 10+75 |
| 4 | core10 + **P3**(직전 5-Lot detrend) | 10+75 |
| 5 | core10 + **P1+P2+P3** | 10+225 |
| 6 | **P4 두단계**(stage1 core10 레벨 → stage2 센서기저 잔차) | 10 / 75 |

**판정 규칙(고정)**
- 스크리닝 = **seed 42 단일**. core10 대비 **−3pt 이상 개선**한 후보만 V3에서 다중시드 재확인.
- **GV2 는 스크리닝이지 최종 판정이 아님.** 최종은 GV3(개선 ≥3pt ∧ R² ≥ 0.9 ≈ RMSE 82.7).
- 6후보 **전부 −3pt 미달 시 → 중단기준(PLAN §7) 충족** → 정직 종료 수순.
- fold 구조는 **V0 와 완전 동일**(확장창·초기 4주·H=7d·MIN_TEST 30) → 후보 #1 이 98.361 을 재현해야 함(하니스 일치 확인).

## 전제 파일
`v13_fdc_pool_wf_oof.csv.gz` · `core10_meta_wf.csv` · `lean_timestable_set.json` · `tuned_params_v13_xgb.json` · `train_data.csv` · **`data/v14_P1.csv`·`v14_P2.csv`·`v14_P3.csv`(V1 산출)** · `v14_features.py`

## 예상 런타임
로컬 venv 기준 **약 3~8분** (총 약 42회 XGB 학습).

## 확인 포인트
- [ ] `[셀3]` 후보별 피처 수 · floor 5센서 각 ≥1(센서 포함 후보) · 누수 assert 통과
- [ ] `[셀5]` **후보#1 core10 = 98.361 재현**(하니스 일치) — 어긋나면 원인 규명 전 판정 금지
- [ ] `[셀7]` 결과표 · Δ(vs core10) · GV2 플래그 · 기여 분해
- [ ] `results/v14_V2_screening.json` 저장

## 산출물
`modeling_v14/results/v14_V2_screening.json` (후보별 pooled RMSE·R²·fold별·Δ) → 강건이 GV2 판정 + REPORT_03 작성.


In [1]:
# [셀1] import + 모듈 + 경로 + 동결 상수 (V0 승계)
import warnings; warnings.filterwarnings("ignore")
import os, sys, json, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error

for _p in [os.getcwd(), os.path.dirname(os.getcwd())]:
    if _p and _p not in sys.path:
        sys.path.insert(0, _p)
import v14_features as vf

FMT = "%Y-%m-%d %H:%M:%S.%f"
ID_COL, TARGET_COL, C20_COL = vf.ID_COL, vf.TARGET_COL, vf.C20_COL
SIGMA_C65 = 261.7
INIT_WEEKS, H_DAYS, MIN_TEST = 4, 7, 30          # V0 와 동일(fold 구조 일치)
BENCH_CORE10 = 98.361                            # V0 로컬 확정치
SEED = 42

def _find(name):
    for d in [".", "data", "colab_GA", "..", os.path.join("..", "modeling_v13"),
              os.path.join("..", "modeling_v13", "data"), os.path.join("..", "modeling_v13", "colab_GA"),
              os.path.join("..", "문제1(하)"), "문제1(하)"]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} 없음")

POOL = _find("v13_fdc_pool_wf_oof.csv.gz"); META = _find("core10_meta_wf.csv")
LEANJSON = _find("lean_timestable_set.json"); XGBJSON = _find("tuned_params_v13_xgb.json"); TRAIN = _find("train_data.csv")

def _rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
def r2_honest(rmse): return round(1 - (rmse / SIGMA_C65) ** 2, 4)
xgj = json.load(open(XGBJSON, encoding="utf-8"))
def make_model(seed=SEED):
    p = dict(xgj["params"]); p.update(objective="reg:squarederror", tree_method="hist",
             device="cpu", random_state=int(seed), n_estimators=int(xgj["n_estimators"]))
    return xgb.XGBRegressor(**p)
print("xgboost", xgb.__version__, "| 벤치마크 core10 =", BENCH_CORE10, "| seed", SEED)

xgboost 3.3.0 | 벤치마크 core10 = 98.361 | seed 42


In [2]:
# [셀2] 로드 + P1/P2/P3 병합 (V1 산출)
pool = pd.read_csv(POOL); pool[ID_COL] = pool[ID_COL].astype(str)
meta = pd.read_csv(META); meta[ID_COL] = meta[ID_COL].astype(str)
df = pool.merge(meta, on=ID_COL, how="inner")
tt = pd.read_csv(TRAIN, usecols=[ID_COL, "C40"]); tt[ID_COL] = tt[ID_COL].astype(str)
tt["_ts"] = pd.to_datetime(tt["C40"], format=FMT)
df = df.merge(tt.groupby(ID_COL)["_ts"].min().reset_index().rename(columns={"_ts": "wf_ts"}),
              on=ID_COL, how="inner").reset_index(drop=True)

n0 = len(df)
PFEATS = {}
for nm in ["P1", "P2", "P3"]:
    fp = os.path.join("data", f"v14_{nm}.csv")
    assert os.path.exists(fp), f"{fp} 없음 — V1 노트북 먼저 실행"
    Pk = pd.read_csv(fp); Pk[ID_COL] = Pk[ID_COL].astype(str)
    PFEATS[nm] = [c for c in Pk.columns if c != ID_COL]
    df = df.merge(Pk, on=ID_COL, how="inner")
assert len(df) == n0, f"병합 후 행수 변화 {n0}->{len(df)} (C64 불일치?)"
y = df[TARGET_COL].to_numpy(float)
print(f"[셀2] df {df.shape} | P1 {len(PFEATS['P1'])} · P2 {len(PFEATS['P2'])} · P3 {len(PFEATS['P3'])} | 행수 유지 OK")

[셀2] df (11939, 893) | P1 75 · P2 75 · P3 75 | 행수 유지 OK


In [3]:
# [셀3] 후보셋 정의 + 규율 assert (floor · 누수)
LEAN = json.load(open(LEANJSON, encoding="utf-8"))["lean_features"]
SENSOR_BASE = vf.sensor_base_cols(LEAN)                  # 원시 센서 집계 75 (P4 stage2 용)
CORE = [c for c in vf.CORE10 if c in df.columns]
assert len(CORE) == 10, f"core10 != 10 ({len(CORE)})"

CANDS = {
    "1_core10":        CORE,
    "2_core10+P1":     CORE + PFEATS["P1"],
    "3_core10+P2":     CORE + PFEATS["P2"],
    "4_core10+P3":     CORE + PFEATS["P3"],
    "5_core10+P1P2P3": CORE + PFEATS["P1"] + PFEATS["P2"] + PFEATS["P3"],
}
BAD = {ID_COL, C20_COL, TARGET_COL, "fold_kf5", "wf_ts", "lot_ts"}
for nm, cols in CANDS.items():
    assert not (set(cols) & BAD), f"R2 누수: {nm} 에 ID/타깃 유입"
    assert all(c in df.columns for c in cols), f"{nm} 누락 컬럼"
    sensor_part = [c for c in cols if c not in CORE]
    if sensor_part:                                       # 센서 포함 후보만 floor 적용
        vf.assert_floor(sensor_part, nm)
vf.assert_floor(SENSOR_BASE, "P4_stage2")
print("[셀3] 후보 피처 수:", {k: len(v) for k, v in CANDS.items()}, "| P4 stage2", len(SENSOR_BASE))
print("[셀3] floor·누수 assert 전부 통과")

[셀3] 후보 피처 수: {'1_core10': 10, '2_core10+P1': 85, '3_core10+P2': 85, '4_core10+P3': 85, '5_core10+P1P2P3': 235} | P4 stage2 75
[셀3] floor·누수 assert 전부 통과


In [4]:
# [셀4] 롤링 하니스 (V0 와 동일 fold 구조)
lot_ts = df.groupby(C20_COL)["wf_ts"].transform("median")
t0 = df["wf_ts"].min(); init_end = t0 + pd.Timedelta(weeks=INIT_WEEKS); end = df["wf_ts"].max()
H = pd.Timedelta(days=H_DAYS)

def _folds():
    T = init_end; out = []
    while T < end:
        te = ((lot_ts > T) & (lot_ts <= T + H)).to_numpy(); ti = np.where(te)[0]
        if len(ti) >= MIN_TEST:
            out.append((T, np.where((lot_ts <= T).to_numpy())[0], ti))
        T = T + H
    return out
FOLDS = _folds()
print(f"[셀4] fold {len(FOLDS)}개 (V0 와 동일해야 함: 6) | 초기창 ~{str(init_end)[:10]}")

def rolling_eval(feat_cols, seed=SEED):
    X = df[feat_cols]; PY_, PP_, fr = [], [], []
    for T, tri, ti in FOLDS:
        m = make_model(seed); m.fit(X.iloc[tri], y[tri]); p = m.predict(X.iloc[ti])
        PY_.append(y[ti]); PP_.append(p); fr.append(round(_rmse(y[ti], p), 2))
    pooled = _rmse(np.concatenate(PY_), np.concatenate(PP_))
    return pooled, r2_honest(pooled), fr

def rolling_eval_p4(core_cols, sensor_cols, seed=SEED):
    Xc, Xs = df[core_cols], df[sensor_cols]; PY_, PP_, fr = [], [], []
    for T, tri, ti in FOLDS:
        pred, _lvl = vf.p4_two_stage_fold(tri, ti, Xc, Xs, y, lambda: make_model(seed))
        PY_.append(y[ti]); PP_.append(pred); fr.append(round(_rmse(y[ti], pred), 2))
    pooled = _rmse(np.concatenate(PY_), np.concatenate(PP_))
    return pooled, r2_honest(pooled), fr

[셀4] fold 6개 (V0 와 동일해야 함: 6) | 초기창 ~2018-12-29


In [5]:
# [셀5] 후보 1~5 채점 + 하니스 일치 확인 (#1 은 98.361 재현해야)
RES = {}
t_all = time.time()
for nm, cols in CANDS.items():
    t0_ = time.time()
    rm, r2, fr = rolling_eval(cols)
    RES[nm] = dict(pooled=round(rm, 3), r2=r2, folds=fr, n_feat=len(cols))
    print(f"  {nm:18} pooled {rm:8.3f}  R² {r2:.4f}  ({time.time()-t0_:.0f}s)")
d_bench = abs(RES["1_core10"]["pooled"] - BENCH_CORE10)
print(f"\n[셀5] 하니스 일치: core10 {RES['1_core10']['pooled']:.3f} vs V0 {BENCH_CORE10} "
      f"|Δ|={d_bench:.3f} → {'OK' if d_bench <= 0.5 else '‼CHECK (원인규명 전 판정 금지)'}")
print(f"[셀5] 총 {time.time()-t_all:.0f}s")

  1_core10           pooled   98.361  R² 0.8587  (3s)
  2_core10+P1        pooled  111.719  R² 0.8178  (4s)
  3_core10+P2        pooled  106.439  R² 0.8346  (4s)
  4_core10+P3        pooled   98.855  R² 0.8573  (3s)
  5_core10+P1P2P3    pooled  107.168  R² 0.8323  (8s)

[셀5] 하니스 일치: core10 98.361 vs V0 98.361 |Δ|=0.000 → OK
[셀5] 총 21s


In [6]:
# [셀6] 후보 6 — P4 두단계
t0_ = time.time()
rm4, r24, fr4 = rolling_eval_p4(CORE, SENSOR_BASE)
RES["6_P4_2stage"] = dict(pooled=round(rm4, 3), r2=r24, folds=fr4, n_feat=f"{len(CORE)}/{len(SENSOR_BASE)}")
print(f"  6_P4_2stage        pooled {rm4:8.3f}  R² {r24:.4f}  ({time.time()-t0_:.0f}s)")

  6_P4_2stage        pooled   99.099  R² 0.8566  (4s)


In [7]:
# [셀7] 결과표 + Δ + GV2 스크리닝 플래그 + 기여 분해 + 저장
base = RES["1_core10"]["pooled"]
rows = []
for nm, r in RES.items():
    imp = round(base - r["pooled"], 3)                       # +면 개선
    rows.append(dict(후보=nm, 피처=r["n_feat"], pooled=r["pooled"], R2=r["r2"],
                     Δvs_core10=imp, 개선_3pt="✅" if imp >= 3.0 else "—",
                     R2_0p9="✅" if r["r2"] >= 0.9 else "—"))
tab = pd.DataFrame(rows)
print("=== [셀7] GV2 스크리닝 결과 (판정은 강건) ===")
print(tab.to_string(index=False))

passers = [r["후보"] for r in rows if r["개선_3pt"] == "✅" and r["후보"] != "1_core10"]
print(f"\n-3pt 통과 후보: {passers if passers else '없음'}")
print("기여 분해:", {nm: round(base - RES[nm]["pooled"], 3)
                    for nm in ["2_core10+P1", "3_core10+P2", "4_core10+P3", "5_core10+P1P2P3", "6_P4_2stage"]})
if not passers:
    print("\n→ 6후보 전부 −3pt 미달: 중단기준(PLAN §7) 충족 신호. 최종 판정은 강건(GV3).")
else:
    print("\n→ 통과 후보 존재: V3 에서 다중시드 재확인 + R²≥0.9 확인 후 GV3 판정.")

os.makedirs("results", exist_ok=True)
out = dict(bench_core10_V0=BENCH_CORE10, seed=SEED, n_folds=len(FOLDS), results=RES,
           passers=passers, generated="V2_screening")
with open(os.path.join("results", "v14_V2_screening.json"), "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=1)
print("\n[셀7] 저장 results/v14_V2_screening.json")
print("=== V2 완료 — 강건에게 결과표 회신 → GV2 판정 + REPORT_03 ===")

=== [셀7] GV2 스크리닝 결과 (판정은 강건) ===
             후보    피처  pooled     R2  Δvs_core10 개선_3pt R2_0p9
       1_core10    10  98.361 0.8587       0.000      —      —
    2_core10+P1    85 111.719 0.8178     -13.358      —      —
    3_core10+P2    85 106.439 0.8346      -8.078      —      —
    4_core10+P3    85  98.855 0.8573      -0.494      —      —
5_core10+P1P2P3   235 107.168 0.8323      -8.807      —      —
    6_P4_2stage 10/75  99.099 0.8566      -0.738      —      —

-3pt 통과 후보: 없음
기여 분해: {'2_core10+P1': -13.358, '3_core10+P2': -8.078, '4_core10+P3': -0.494, '5_core10+P1P2P3': -8.807, '6_P4_2stage': -0.738}

→ 6후보 전부 −3pt 미달: 중단기준(PLAN §7) 충족 신호. 최종 판정은 강건(GV3).

[셀7] 저장 results/v14_V2_screening.json
=== V2 완료 — 강건에게 결과표 회신 → GV2 판정 + REPORT_03 ===
